# BioSEL PAM Analysis Results - Interactive Exploration

Load pre-computed results and visualize PAM actuation metrics.

**Prerequisites:**
1. Run: `uv run python Code/BioSEL-PULSE/core/run_analysis.py Curated_Data`
2. This notebook auto-detects the results in `analysis_results/` (your regenerated run) or the provided `Dataset/` folder

**Usage:** Run all cells top to bottom. Use the single **Dataset** dropdown to switch datasets across all figures simultaneously.

In [ ]:
import pickle
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import pandas as pd
from matplotlib.gridspec import GridSpec
import ipywidgets as widgets
from IPython.display import display

_CMAP = "magma"
_DERIVED_COLS = ("Velocity_Contraction", "Velocity_Pressure", "Acceleration_Contraction")

def _setpoint_twin(ax, t, setpoint, alpha=0.55):
    """Overlay Setpoint as a step signal on a right twin-axis."""
    color = "#AAAAAA"
    ax2 = ax.twinx()
    ax2.step(t, setpoint, where="post", color=color, alpha=alpha, linewidth=1.0)
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(["0", "1"], color=color, fontsize=7)
    ax2.set_ylabel("Setpoint", color=color, fontsize=8)
    ax2.spines["right"].set_color(color)
    ax2.tick_params(axis="y", colors=color)

def _add_colorbar(fig, ax, n_cycles):
    sm = cm.ScalarMappable(cmap=_CMAP, norm=plt.Normalize(vmin=1, vmax=n_cycles))
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label=f"Cycle (1–{n_cycles})", shrink=0.8)

# Custom pickle file (optional)
customPKL = None  # pkl filename override - e.g. "MyRun.pkl" (default: all_results.pkl)
customDIR = None  # For when you dont need to path it in

pkl_filename = customPKL or "all_results.pkl"

all_results = {}
dataset_options = []

def _load_pkl(base_path):
    global all_results, dataset_options
    p = Path(base_path) / pkl_filename
    if not p.exists():
        print(f"Error: {p} not found")
        print("Please run: uv run python Code/BioSEL-PULSE/core/run_analysis.py [path to data]")
        all_results = {}
        dataset_options = []
        return
    with open(p, "rb") as f:
        all_results = pickle.load(f)
    dataset_options = list(all_results.keys())
    print(f"✅ Loaded {len(all_results)} datasets from {base_path}")

if customDIR:
    _load_pkl(customDIR)
else:
    _candidates = [
        Path("../analysis_results"),       # your regenerated run, cwd = explore/ (classic Jupyter/JupyterLab)
        Path("analysis_results"),          # your regenerated run, cwd = BioSEL-PULSE/ (VS Code)
        Path("../../Dataset"),             # provided Zenodo PKL, cwd = BioSEL-PULSE/
        Path("../../../Dataset"),          # provided Zenodo PKL, cwd = explore/
    ]
    _existing = [p for p in _candidates if p.is_dir()]

    if not _existing:
        _load_pkl(_candidates[0])  # triggers not-found error message
    elif len(_existing) == 1:
        _load_pkl(_existing[0])
    else:
        _source_out = widgets.Output()
        _source_picker = widgets.RadioButtons(
            options=[str(p) for p in _existing],
            value=str(_existing[0]),
            description="PKL source:",
            style={"description_width": "initial"},
        )

        def _on_source_change(change):
            with _source_out:
                _source_out.clear_output()
                _load_pkl(change["new"])

        _source_picker.observe(_on_source_change, names="value")
        display(widgets.VBox([
            widgets.HTML("<b>Multiple result sources found - select one:</b>"),
            _source_picker,
            _source_out,
        ]))
        _load_pkl(_existing[0])  # auto-load first match immediately

In [ ]:
# -- Single shared dataset selector - controls all figures below ---------------
shared_dataset = widgets.Dropdown(
    options=dataset_options,
    value=dataset_options[0],
    description="Dataset:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="60%")
)
display(widgets.VBox([
    widgets.HTML("<b style='font-size:14px'>📂 Select Dataset (updates all figures)</b>"),
    shared_dataset
]))

## 1. Active Data: Pressure, Displacement, Hysteresis Loop

In [ ]:
def plot_active_data(dataset):
    data = all_results[dataset]
    dat = data["activeDat"]
    x = dat["Pressure"].values
    y = dat["Contraction"].values
    time = dat["Time_s"].values

    fig1 = plt.figure(figsize=(12, 10))
    fig1.suptitle(f"Active Data - {dataset}", fontsize=14, fontweight="bold")
    gs = GridSpec(3, 1, figure=fig1, hspace=0.35)

    ax1 = fig1.add_subplot(gs[0])
    ax1.plot(time, x, linewidth=1.2)
    ax1.set_xlabel("Time (s)")
    ax1.set_ylabel("Pressure (kPa)")
    ax1.set_title("Input - Pressure vs Time")
    ax1.grid(True, alpha=0.3)

    ax2 = fig1.add_subplot(gs[1])
    ax2.plot(time, y, linewidth=1.2, color="#D65E3E")
    ax2.set_xlabel("Time (s)")
    ax2.set_ylabel("Contraction (mm)")
    ax2.set_title("Output - Contraction vs Time")
    ax2.grid(True, alpha=0.3)

    ax3 = fig1.add_subplot(gs[2])
    ax3.plot(x, y, linewidth=1.2, color="#78B131")
    ax3.set_xlabel("Pressure (kPa)")
    ax3.set_ylabel("Contraction (mm)")
    ax3.set_title("Hysteresis Loop")
    ax3.grid(True, alpha=0.3)

    # plt.tight_layout()
    plt.show()
    print(f"📊 {dataset}: {len(time):,} samples, {time.max():.1f}s duration")

out_active = widgets.interactive_output(plot_active_data, {"dataset": shared_dataset})
display(out_active)

## 2. Centroid Drift

In [ ]:
def plot_centroid_drift(dataset):
    data = all_results[dataset]
    x_c = data["CentroidDrift_results"]["Centroid_Pressure"].values
    y_c = data["CentroidDrift_results"]["Centroid_Pot"].values
    ell_area = data["ellipse_result"]["area"]

    fig2 = plt.figure(figsize=(10, 8))
    fig2.suptitle(f"Centroid Drift - {dataset}", fontsize=14, fontweight="bold")
    ax = fig2.add_subplot(111)

    ax.scatter(x_c, y_c, s=80, alpha=0.6, label="Cycle Centroids")
    cx_mean = data["CentroidDrift_summary"][0]["mean"]
    cy_mean = data["CentroidDrift_summary"][1]["mean"]
    ax.scatter([cx_mean], [cy_mean], s=140, marker="D", color="red", label="Cloud Center", zorder=5)

    ax_lim = ax.get_xlim()
    xfit = np.linspace(ax_lim[0], ax_lim[1], 100)
    mdl = data["mdl_fit"]
    yfit = mdl.predict(xfit.reshape(-1, 1))
    ax.plot(xfit, yfit, "--k", linewidth=1.8, alpha=0.5, label="Linear Fit")

    ell_coords = data["ellipse_result"]["ellipse_coords"].copy()
    ax.plot(ell_coords[:, 0], ell_coords[:, 1], "-r", linewidth=2.5, alpha=0.7, label="95% Ellipse")

    ax.set_xlabel("Centroid Pressure (kPa)")
    ax.set_ylabel("Centroid Displacement (mm)")
    ax.set_title(f"Centroid Drift | Ellipse Area = {ell_area:.4f}")
    ax.legend(loc="best")
    ax.grid(True, alpha=0.3)

    # plt.tight_layout()
    plt.show() # Commented out, was noticing that this was causing issues for initial rendering, left alone just in case

out_centroid = widgets.interactive_output(plot_centroid_drift, {"dataset": shared_dataset})
display(out_centroid)

## 3. Hysteresis Loop - By Cycle (Interactive Slider)

In [ ]:
max_cycles = {}
for ds_name, data in all_results.items():
    if "Cycle" in data["activeDat"].columns:
        max_cycles[ds_name] = int(data["activeDat"]["Cycle"].max())
    else:
        max_cycles[ds_name] = 1

def plot_cycle_hysteresis(dataset, cycle_num):
    data = all_results[dataset]
    dat = data["activeDat"]

    if "Cycle" not in dat.columns:
        print(f"WARNING: Dataset '{dataset}' does not have cycle numbers")
        return

    max_cyc = max_cycles[dataset]
    if cycle_num > max_cyc:
        print(f"Cycle {cycle_num} exceeds max cycle {max_cyc} for this dataset")
        return

    cycle_data = dat[dat["Cycle"] == cycle_num]
    if len(cycle_data) == 0:
        print(f"No data for cycle {cycle_num}")
        return

    x = cycle_data["Pressure"].values
    y = cycle_data["Contraction"].values
    mean_loop_area = data["LoopMean"]

    fig = plt.figure(figsize=(8, 6))
    fig.suptitle(f"Hysteresis Loop - {dataset} | Cycle {cycle_num}", fontsize=12, fontweight="bold")
    ax = fig.add_subplot(111)

    ax.plot(x, y, linewidth=2.5, marker="o", markersize=4, label=f"Cycle {cycle_num}")
    ax.set_xlabel("Pressure (kPa)")
    ax.set_ylabel("Contraction (mm)")
    ax.grid(True, alpha=0.3)
    ax.legend()

    cycle_area = 0.5 * float(abs(sum(x[i]*y[(i+1) % len(x)] - x[(i+1) % len(x)]*y[i] for i in range(len(x)))))
    deviation = ((cycle_area - mean_loop_area) / mean_loop_area * 100) if mean_loop_area != 0 else 0

    info_text = f"Cycle Area: {cycle_area:.4f} | Mean: {mean_loop_area:.4f} | Deviation: {deviation:+.1f}%"
    ax.text(0.5, -0.12, info_text, transform=ax.transAxes, ha="center", fontsize=9, style="italic")

    plt.tight_layout()
    plt.show()

cycle_slider = widgets.IntSlider(
    min=1, max=max_cycles[dataset_options[0]], step=1, value=1, description="Cycle:"
)
cycle_left_btn = widgets.Button(description="←", tooltip="Previous cycle")
cycle_right_btn = widgets.Button(description="→", tooltip="Next cycle")

def _on_ds_change(change):
    max_cyc = max_cycles[change["new"]]
    cycle_slider.max = max_cyc
    cycle_slider.value = 1

def _on_left(b):
    if cycle_slider.value > 1:
        cycle_slider.value -= 1

def _on_right(b):
    if cycle_slider.value < cycle_slider.max:
        cycle_slider.value += 1

shared_dataset.observe(_on_ds_change, names="value")
cycle_left_btn.on_click(_on_left)
cycle_right_btn.on_click(_on_right)

out_cycle = widgets.interactive_output(
    plot_cycle_hysteresis, {"dataset": shared_dataset, "cycle_num": cycle_slider}
)
display(widgets.VBox([
    widgets.HBox([cycle_left_btn, cycle_slider, cycle_right_btn]),
    out_cycle
]))

## 4. Summary Table (All Datasets)

## 5. Raw Signals (FullDat)

Full time-window pressure and contraction signals. Setpoint overlaid as a step signal on a right twin-axis when available.

In [ ]:
def plot_raw_signals(dataset):
    data = all_results[dataset]
    full_dat = data.get("FullDat")
    if full_dat is None or full_dat.empty:
        print(f"⚠️  FullDat missing for '{dataset}'")
        return

    fig, (ax_p, ax_c) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{dataset}  -  Raw Signals", fontsize=12, fontweight="bold")

    t_all = full_dat["Time_s"].values
    ax_p.plot(t_all, full_dat["Pressure"].values, linewidth=1.5, color="#1f77b4")
    ax_c.plot(t_all, full_dat["Contraction"].values, linewidth=1.5, color="#d62728")

    ax_p.set(xlabel="Time (s)", ylabel="Pressure (kPa)", title="Pressure vs Time")
    ax_p.grid(True, alpha=0.4)
    ax_c.set(xlabel="Time (s)", ylabel="Contraction (mm)", title="Contraction vs Time")
    ax_c.grid(True, alpha=0.4)

    if "Setpoint" in full_dat.columns:
        _setpoint_twin(ax_p, t_all, full_dat["Setpoint"].values)
        _setpoint_twin(ax_c, t_all, full_dat["Setpoint"].values)

    fig.tight_layout()
    plt.show()

out_raw = widgets.interactive_output(plot_raw_signals, {"dataset": shared_dataset})
display(out_raw)

## 6. Derived Metrics

Pressure rise rate, contraction velocity, and contraction acceleration over time. All cycles overlaid, colormap-coded early→late. Only visible when `derived_metrics: true`.

In [ ]:
def plot_derived_metrics(dataset):
    data = all_results[dataset]
    active_dat = data.get("activeDat")
    if active_dat is None or active_dat.empty or not all(c in active_dat.columns for c in _DERIVED_COLS):
        print(f"ℹ️  No derived metrics for '{dataset}' - add derived_metrics: true to .metadata.yaml and re-run.")
        return

    act_cycles = sorted(active_dat["Cycle"].dropna().unique())
    n_act = len(act_cycles)
    act_colors = plt.get_cmap(_CMAP)(np.linspace(0, 1, max(n_act, 1)))

    fig, (ax_vp, ax_vc, ax_ac) = plt.subplots(1, 3, figsize=(15, 6), layout="constrained")
    fig.suptitle(f"{dataset}  -  Derived Metrics", fontsize=12, fontweight="bold")

    for i, cyc in enumerate(act_cycles):
        cyc_df = active_dat[active_dat["Cycle"] == cyc]
        t_abs = cyc_df["Time_s"].values
        col = act_colors[i]
        ax_vp.plot(t_abs, cyc_df["Velocity_Pressure"].values,        color=col, linewidth=1.5, alpha=0.7)
        ax_vc.plot(t_abs, cyc_df["Velocity_Contraction"].values,     color=col, linewidth=1.5, alpha=0.7)
        ax_ac.plot(t_abs, cyc_df["Acceleration_Contraction"].values, color=col, linewidth=1.5, alpha=0.7)

    ax_vp.set(xlabel="Time (s)", ylabel="dPressure/dt (kPa/s)",      title="Pressure Rise Rate")
    ax_vp.grid(True, alpha=0.3)
    ax_vc.set(xlabel="Time (s)", ylabel="dContraction/dt (mm/s)",    title="Contraction Velocity")
    ax_vc.grid(True, alpha=0.3)
    ax_ac.set(xlabel="Time (s)", ylabel="d²Contraction/dt² (mm/s²)", title="Contraction Acceleration")
    ax_ac.grid(True, alpha=0.3)

    if "Setpoint" in active_dat.columns:
        t_act = active_dat["Time_s"].values
        sp = active_dat["Setpoint"].values
        _setpoint_twin(ax_vp, t_act, sp, alpha=0.2)
        _setpoint_twin(ax_vc, t_act, sp, alpha=0.2)
        _setpoint_twin(ax_ac, t_act, sp, alpha=0.2)

    sm = cm.ScalarMappable(cmap=_CMAP, norm=plt.Normalize(vmin=1, vmax=n_act))
    sm.set_array([])
    fig.colorbar(sm, ax=[ax_vp, ax_vc, ax_ac], orientation="horizontal",
                 label=f"Cycle (1–{n_act})", shrink=0.4, pad=0.12)
    plt.show()

out_derived = widgets.interactive_output(plot_derived_metrics, {"dataset": shared_dataset})
display(out_derived)

## 7. Hysteresis & Phase Portrait

Global hysteresis loop (FullDat, all cycles) alongside the contraction velocity phase portrait. Cycles colormap-coded early→late. Only visible when `derived_metrics: true`.

In [ ]:
def plot_hysteresis_phase(dataset):
    data = all_results[dataset]
    full_dat = data.get("FullDat")
    active_dat = data.get("activeDat")

    if full_dat is None or full_dat.empty:
        print(f"⚠️  FullDat missing for '{dataset}'")
        return
    if active_dat is None or active_dat.empty or not all(c in active_dat.columns for c in _DERIVED_COLS):
        print(f"ℹ️  No derived metrics for '{dataset}' - add derived_metrics: true to .metadata.yaml and re-run.")
        return

    cycles = sorted(full_dat["Cycle"].dropna().unique())
    n_cyc = len(cycles)
    colors = plt.get_cmap(_CMAP)(np.linspace(0, 1, max(n_cyc, 1)))

    act_cycles = sorted(active_dat["Cycle"].dropna().unique())
    n_act = len(act_cycles)
    act_colors = plt.get_cmap(_CMAP)(np.linspace(0, 1, max(n_act, 1)))

    fig, (ax_h, ax_ph) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{dataset}  -  Hysteresis & Phase Portrait", fontsize=12, fontweight="bold")

    for i, cyc in enumerate(cycles):
        cyc_df = full_dat[full_dat["Cycle"] == cyc]
        ax_h.plot(cyc_df["Pressure"].values, cyc_df["Contraction"].values,
                  color=colors[i], linewidth=1.2, alpha=0.7)

    for i, cyc in enumerate(act_cycles):
        cyc_df = active_dat[active_dat["Cycle"] == cyc]
        ax_ph.plot(cyc_df["Contraction"].values, cyc_df["Velocity_Contraction"].values,
                   color=act_colors[i], linewidth=1.2, alpha=0.7)

    ax_h.set(xlabel="Pressure (kPa)", ylabel="Contraction (mm)", title="Global Hysteresis Loop")
    ax_h.grid(True, alpha=0.3)
    ax_ph.set(xlabel="Contraction (mm)", ylabel="Contraction Velocity (mm/s)", title="Velocity Phase Portrait")
    ax_ph.grid(True, alpha=0.3)

    _add_colorbar(fig, ax_h, n_cyc)
    _add_colorbar(fig, ax_ph, n_act)
    fig.tight_layout()
    plt.show()

out_phase = widgets.interactive_output(plot_hysteresis_phase, {"dataset": shared_dataset})
display(out_phase)

## 8. Raw vs Filtered Hysteresis

Side-by-side comparison of the active hysteresis loop before and after filtering. Only visible when `derived_metrics: true` is set in `.metadata.yaml`.

In [ ]:
def plot_raw_vs_filtered(dataset):
    data = all_results[dataset]
    active_dat = data.get("activeDat")
    if active_dat is None or active_dat.empty or not all(c in active_dat.columns for c in _DERIVED_COLS):
        print(f"ℹ️  No derived metrics for '{dataset}' - add derived_metrics: true to .metadata.yaml and re-run.")
        return

    act_cycles = sorted(active_dat["Cycle"].dropna().unique())
    n_act = len(act_cycles)
    act_colors = plt.get_cmap(_CMAP)(np.linspace(0, 1, max(n_act, 1)))

    fig, (ax_raw, ax_filt) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f"{dataset}  -  Raw vs Filtered Hysteresis", fontsize=12, fontweight="bold")

    for i, cyc in enumerate(act_cycles):
        cyc_df = active_dat[active_dat["Cycle"] == cyc]
        ax_raw.plot(cyc_df["Pressure"].values, cyc_df["Contraction"].values,
                    color=act_colors[i], linewidth=1.2, alpha=0.7)
        ax_filt.plot(cyc_df["Pressure_filt"].values, cyc_df["Contraction_filt"].values,
                     color=act_colors[i], linewidth=1.2, alpha=0.7)

    ax_raw.set(xlabel="Pressure (kPa)", ylabel="Contraction (mm)", title="Active Hysteresis (raw)")
    ax_raw.grid(True, alpha=0.3)
    ax_filt.set(xlabel="Pressure filtered (kPa)", ylabel="Contraction filtered (mm)", title="Active Hysteresis (filtered)")
    ax_filt.grid(True, alpha=0.3)

    _add_colorbar(fig, ax_raw, n_act)
    _add_colorbar(fig, ax_filt, n_act)
    fig.tight_layout()
    plt.show()

out_filtered = widgets.interactive_output(plot_raw_vs_filtered, {"dataset": shared_dataset})
display(out_filtered)

In [ ]:
summary_rows = []
for ds_name, data in all_results.items():
    summary_rows.append({
        "Dataset": ds_name,
        "LoopMean": data["LoopMean"],
        "NormLoopValue": data["NormLoopValue"],
        "EllipseArea": data["ellipse_result"]["area"],
        "Eccentricity": data["ellipse_result"]["eccentricity"],
    })

summary_df = pd.DataFrame(summary_rows)
print("\n" + "="*100)
print("SUMMARY TABLE - All Datasets")
print("="*100)
print(summary_df.to_string(index=False))